## The Service abstraction — stable identity by label selector

A **Service** is an object with two halves: a **virtual IP + name** (clients connect here; it never changes) and a **selector** (`app: api`, the label query from notebook 02). A controller — the **endpoints controller** — runs the loop:

1. List all pods matching the selector that are **Ready**.
2. Write their IPs into an `Endpoints` / `EndpointSlice` object.
3. Every node's `kube-proxy` watches those endpoints and programs the node so the Service's VIP forwards to a ready pod IP.

The minimum manifest:

```yaml
apiVersion: v1
kind: Service
metadata: { name: api }
spec:
  selector: { app: api }   # picks any pod with this label
  ports:
    - port: 80             # the port the Service exposes
      targetPort: 5678     # the port the container listens on
```

Notice what's *not* there: **no list of pod IPs** (zero hardcoded references), **no type** (defaults to `ClusterIP`), **no image or replicas** (a Service runs nothing — it only routes).

Apply it and the cluster does three things behind your back:

- Allocates a **ClusterIP** from the service CIDR (typically `10.96.0.0/12`).
- Registers a **DNS name** — `api.<namespace>.svc.cluster.local` → the ClusterIP.
- Computes the **Endpoints** for the selector and pushes them to every `kube-proxy`.

From then on, anything in the cluster that knows the name `api` can reach it. The elegance: the Service never names a pod — it names a *label*, and the cluster re-resolves that live. On our map this is the **Service** chip resolving onto the **Pods** box.